# Extraction of landscape geometry features


This notebook explores preprocessing, local geometric features, terrain extraction, and surface-area estimation for a mobile laser scanning point cloud.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import open3d as o3d
import seaborn as sns
from scipy import spatial, stats

In [ ]:
sns.set_theme()


def find_repo_root(start: Path | None = None) -> Path:
    """Find the repository root from the current working directory."""
    start = (start or Path.cwd()).resolve()
    for directory in (start, *start.parents):
        if (directory / "pyproject.toml").is_file():
            return directory
    msg = "Could not find the repository root containing pyproject.toml"
    raise FileNotFoundError(msg)


REPO_ROOT = find_repo_root()

## Load the original data

Obtained from F. Poux's [video tutorial](https://www.youtube.com/watch?v=WKSJcG97gE4) on 3D point cloud feature extraction, available on [Google Drive](https://drive.google.com/drive/folders/1fwhE5OphpeW4RR0RY8W2jbqmlf5LH6dX).


In [ ]:
data_path = REPO_ROOT / "data" / "UTWENTE" / "MLS_UTWENTE_super_sample.ply"
pcd = o3d.io.read_point_cloud(str(data_path))

In [ ]:
o3d.visualization.draw_geometries([pcd])

## Preprocessing


In [ ]:
# Translation
zmin = pcd.get_min_bound()[2]
trans = np.array([0, 0, -zmin])
pcd = pcd.translate(trans)

In [ ]:
# Color point elevations
points = np.asarray(pcd.points)
elev = points[:, 2]
cmap = sns.color_palette(palette="viridis", as_cmap=True)
norm = plt.Normalize()
color = cmap(norm(elev))[:, :-1]
pcd.colors = o3d.utility.Vector3dVector(color)

In [ ]:
o3d.visualization.draw_geometries([pcd])

## Exploring the unstructured data with an octree


In [ ]:
octree = o3d.geometry.Octree(max_depth=7)
octree.convert_from_point_cloud(pcd, size_expand=0.01)

In [ ]:
o3d.visualization.draw_geometries([octree])

## Downsampling the point cloud using a voxel grid


In [ ]:
pcd_ds = pcd.voxel_down_sample(voxel_size=0.8)  # approximately 10x downsampling

In [ ]:
o3d.visualization.draw_geometries([pcd_ds])

## Removing outliers


In [ ]:
def display_inlier_outlier(
    point_cloud: o3d.geometry.PointCloud | np.ndarray,
    indices: list[int] | np.ndarray,
) -> None:
    """Visualize inliers and outliers in a point cloud."""
    if not isinstance(point_cloud, o3d.geometry.PointCloud):
        converted = o3d.geometry.PointCloud()
        converted.points = o3d.utility.Vector3dVector(np.asarray(point_cloud))
        point_cloud = converted
    inlier = point_cloud.select_by_index(indices)
    outlier = point_cloud.select_by_index(indices, invert=True)
    outlier.paint_uniform_color([1, 0, 0])
    inlier.paint_uniform_color([0.8, 0.8, 0.8])
    o3d.visualization.draw_geometries([inlier, outlier])

In [ ]:
pcd_stat, ind = pcd_ds.remove_statistical_outlier(nb_neighbors=30, std_ratio=3)

In [ ]:
display_inlier_outlier(pcd_ds, ind)

In [ ]:
pcd_rad, ind = pcd_ds.remove_radius_outlier(nb_points=25, radius=5)

In [ ]:
display_inlier_outlier(pcd_ds, ind)

In [ ]:
o3d.visualization.draw_geometries([pcd_rad])

## Final cleaning touches on the point cloud


In [ ]:
# https://www.open3d.org/html/tutorial/Advanced/interactive_visualization.html
# o3d.visualization.draw_geometries_with_editing([pcd_rad])


In [ ]:
data_path = REPO_ROOT / "data" / "UTWENTE" / "MLS_UTWENTE_super_sample_crop.ply"
pcd_crop = o3d.io.read_point_cloud(str(data_path))

In [ ]:
o3d.visualization.draw_geometries([pcd_crop])

## Extracting geometric features


In [ ]:
points = np.asarray(pcd_crop.points)

### In 3D


In [ ]:
def pca(points: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return PCA eigenvalues and eigenvectors in descending order."""
    centered = points - np.mean(points, axis=0)
    covariance = np.cov(centered, rowvar=False)
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    order = np.argsort(eigenvalues)[::-1]
    return eigenvalues[order], eigenvectors[:, order]

In [ ]:
def extract_features(
    eigenvalues: np.ndarray,
    eigenvectors: np.ndarray,
) -> dict[str, float]:
    """Return PCA-based geometric features.

    See https://doi.org/10.5194/isprsannals-II-5-W2-313-2013.
    """
    planarity = (eigenvalues[1] - eigenvalues[2]) / eigenvalues[0]
    linearity = (eigenvalues[0] - eigenvalues[1]) / eigenvalues[0]
    omnivariance = float(np.prod(eigenvalues) ** (1 / 3))
    normal = eigenvectors[:, -1]
    verticality = 1 - abs(normal[2])
    return {
        "planarity": float(planarity),
        "linearity": float(linearity),
        "omnivariance": omnivariance,
        "verticality": float(verticality),
        "nx": float(normal[0]),
        "ny": float(normal[1]),
        "nz": float(normal[2]),
    }

In [ ]:
tree = spatial.KDTree(points)
dist, ind = tree.query(points, k=25)
nbhd = points[ind]

In [ ]:
# Example for a single point
sel = 0
evalues, evectors = pca(nbhd[sel])
feats = extract_features(evalues, evectors)

In [ ]:
feats

In [ ]:
# Surface normals
n = np.empty_like(points)
for i in range(points.shape[0]):
    evalues, evectors = pca(nbhd[i, ...])
    feats = extract_features(evalues, evectors)
    n[i, :] = [feats["nx"], feats["ny"], feats["nz"]]
pcd_crop.normals = o3d.utility.Vector3dVector(n)
pcd_crop.orient_normals_consistent_tangent_plane(20)

In [ ]:
o3d.visualization.draw_geometries([pcd_crop], point_show_normal=True)

### 2D


In [ ]:
def display_selection(
    point_cloud: o3d.geometry.PointCloud | np.ndarray,
    indices: list[int] | np.ndarray,
) -> None:
    """Show selected points in red and unselected points in gray."""
    if not isinstance(point_cloud, o3d.geometry.PointCloud):
        converted = o3d.geometry.PointCloud()
        converted.points = o3d.utility.Vector3dVector(np.asarray(point_cloud))
        point_cloud = converted
    selected = point_cloud.select_by_index(indices)
    unselected = point_cloud.select_by_index(indices, invert=True)
    selected.paint_uniform_color([1, 0, 0])
    unselected.paint_uniform_color([0.8, 0.8, 0.8])
    o3d.visualization.draw_geometries([selected, unselected])

In [ ]:
tree_2d = spatial.KDTree(points[:, :2])
ind_2d = tree_2d.query_ball_point(points[:, :2], 4)

In [ ]:
# Example for a single selection
sel = 0
points_sel = points[ind_2d[sel]]

In [ ]:
display_selection(points, ind_2d[sel])

In [ ]:
# Create a distribution of local elevation ranges.
elevation_ranges = [np.ptp(points[indices, 2]) for indices in ind_2d]

In [ ]:
kernel = stats.gaussian_kde(elevation_ranges)

In [ ]:
# Plot elevation distribution
fig, ax = plt.subplots()
y, bins, patches = ax.hist(
    elevation_ranges,
    bins="fd",
    density=True,
    cumulative=False,
    histtype="bar",
    align="mid",
    orientation="vertical",
    label="measured data",
)
ax.plot(bins, kernel(bins), label="kernel density estimate")
ax.set(xlabel="elevation (m)", ylabel="density")
ax.legend();

In [ ]:
kernel.integrate_box_1d(min(elevation_ranges), max(elevation_ranges))

In [ ]:
kernel.integrate_box_1d(10, 15)

In [ ]:
kernel.covariance

In [ ]:
kernel.covariance_factor()

## Extracting the flat terrain


In [ ]:
def display_plane(
    point_cloud: o3d.geometry.PointCloud | np.ndarray,
    indices: list[int] | np.ndarray,
) -> None:
    """Show planar points in red and non-planar points in gray."""
    if not isinstance(point_cloud, o3d.geometry.PointCloud):
        converted = o3d.geometry.PointCloud()
        converted.points = o3d.utility.Vector3dVector(np.asarray(point_cloud))
        point_cloud = converted
    planar = point_cloud.select_by_index(indices)
    nonplanar = point_cloud.select_by_index(indices, invert=True)
    planar.paint_uniform_color([1, 0, 0])
    nonplanar.paint_uniform_color([0.8, 0.8, 0.8])
    o3d.visualization.draw_geometries([planar, nonplanar])

In [ ]:
o3d.utility.random.seed(42)

In [ ]:
plane_model, ind = pcd_crop.segment_plane(
    distance_threshold=0.5,
    ransac_n=3,
    num_iterations=1000,
)

In [ ]:
a, b, c, d = plane_model
print(f"implicit eqn. ({a:.2e}) x + ({b:.2e}) y + ({c:.2e}) z + ({d:.2e}) = 0")

In [ ]:
display_plane(pcd_crop, ind)

## Estimating the surface area of the flat terrain


In [ ]:
pcd_flat = pcd_crop.select_by_index(ind)
pcd_flat.paint_uniform_color(sns.color_palette()[0])
obb = pcd_flat.get_oriented_bounding_box()
obb.color = sns.color_palette()[1]

In [ ]:
o3d.visualization.draw_geometries([pcd_flat, obb])

The surface-area estimate does not account for the elevation of the flat terrain.


In [ ]:
surface_area = obb.extent[0] * obb.extent[1]
surface_area